# Phase 3 — Model Engineering

This notebook implements the Phase 3 modelling plan defined in `project_specification_phase3_updated.md`. The objective is to predict continuous `price` using `availability`, `rating`, and `category`. `title` and `upc` are excluded as predictors.

The workflow uses an 80/20 split with `random_state=42`, 5-fold K-Fold cross-validation on the training set, leakage-safe scikit-learn pipelines, a mean `DummyRegressor` baseline, Multiple Linear Regression, tuned Random Forest Regression, and tuned SVR. Model selection is based on the lowest mean cross-validated RMSE. The untouched test set is evaluated only after selection.

## 1. Imports and configuration

In [1]:
from pathlib import Path
import sys
import joblib
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.pipeline import FEATURES, NUMERIC_FEATURES, CATEGORICAL_FEATURES, RANDOM_STATE
from src.models.train import load_data, train_models

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "books_processed_data.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "final_price_model.joblib"
RESULTS_DIR = PROJECT_ROOT / "reports"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Data path:", DATA_PATH)
print("Model path:", MODEL_PATH)

Data path: c:\Users\Tan Kai Shi\Documents\Personal Project\Data Science Project 1\data\processed\books_processed_data.csv
Model path: c:\Users\Tan Kai Shi\Documents\Personal Project\Data Science Project 1\models\final_price_model.joblib


## 2. Load and validate the Phase 2 processed dataset

Phase 2 specifies 1,000 observations and six variables: `title`, `price`, `availability`, `rating`, `category`, and `upc`. The `category` variable remains available for transformation in Phase 3.

In [2]:
df = load_data(DATA_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Category count:", df["category"].nunique())
print("Missing values:")
display(df.isna().sum().to_frame("missing"))
print("\nData types:")
display(df.dtypes.to_frame("dtype"))

Shape: (1000, 6)
Columns: ['title', 'price', 'availability', 'rating', 'category', 'upc']
Category count: 50
Missing values:


,missing
title,0
price,0
availability,0
rating,0
category,0
upc,0



Data types:


,dtype
title,str
price,float64
availability,int64
rating,int64
category,str
upc,str


## 3. Define target and modelling features

Target: `price`. Predictors: `availability`, `rating`, and `category`. `title` is excluded as free text and `upc` is excluded as an identifier, following the specification.

In [3]:
TARGET = "price"
print("Target:", TARGET)
print("Features:", FEATURES)
print("Excluded predictors:", ["title", "upc"])
assert FEATURES == ["availability", "rating", "category"]

Target: price
Features: ['availability', 'rating', 'category']
Excluded predictors: ['title', 'upc']


## 4. Train/test split and leakage-safe modelling

The training set contains 80% of observations and the test set contains 20%. `random_state=42` is fixed. All transformations are inside the model pipelines so that they are fitted within each training/CV context.

In [4]:
result = train_models(df)
X_train, X_test = result["X_train"], result["X_test"]
y_train, y_test = result["y_train"], result["y_test"]

print("Training observations:", len(X_train))
print("Test observations:", len(X_test))
print("CV folds:", result["cv"].n_splits)
print("Random state:", RANDOM_STATE)

Training observations: 800
Test observations: 200
CV folds: 5
Random state: 42


## 5. Cross-validated model comparison

RMSE is the primary criterion. The tuned Random Forest and SVR use `GridSearchCV` with 5-fold K-Fold CV and negative RMSE scoring. Linear Regression is not tuned. The baseline is included as the mean-prediction reference.

In [5]:
cv_summary = result["cv_summary"]
display(cv_summary)

selected_name = result["selected_name"]
print("Selected model based on lowest mean CV RMSE:", selected_name)

,Model,Mean_CV_RMSE
0,DummyRegressor,14.426603
1,SVR,14.510834
2,RandomForestRegressor,14.752873
3,LinearRegression,15.151361


Selected model based on lowest mean CV RMSE: DummyRegressor


## 6. Hyperparameter tuning results

In [6]:
tuning_summary = result["tuning_summary"]
display(tuning_summary)

print("Random Forest best parameters:")
print(result["rf_search"].best_params_)
print("\nSVR best parameters:")
print(result["svr_search"].best_params_)

,Model,Best_CV_RMSE,Best_Params
0,RandomForestRegressor,14.752873,"{'model__max_depth': 10, 'model__min_samples_l..."
1,SVR,14.510834,"{'model__C': 1, 'model__epsilon': 0.5, 'model_..."


Random Forest best parameters:
{'model__max_depth': 10, 'model__min_samples_leaf': 2, 'model__min_samples_split': 5, 'model__n_estimators': 200}

SVR best parameters:
{'model__C': 1, 'model__epsilon': 0.5, 'model__gamma': 'auto', 'model__kernel': 'rbf'}


## 7. Final test-set evaluation

The test set has remained untouched during model fitting, preprocessing fitting, cross-validation, hyperparameter tuning, and model selection. It is evaluated only now for the selected model.

In [7]:
test_summary = result["test_summary"]
display(test_summary)

metrics = result["selected_test_metrics"]
print(f"Final selected model: {selected_name}")
print(f"Test RMSE: {metrics['RMSE']:.6f}")
print(f"Test MAE:  {metrics['MAE']:.6f}")
print(f"Test R2:   {metrics['R2']:.6f}")

,Model,RMSE,MAE,R2
0,DummyRegressor,14.530465,12.661919,-0.005098


Final selected model: DummyRegressor
Test RMSE: 14.530465
Test MAE:  12.661919
Test R2:   -0.005098


## 8. Inspect the final fitted pipeline

The saved object is a complete preprocessing-and-model pipeline, allowing new observations to be transformed consistently before prediction.

In [8]:
final_model = result["selected_model"]
print(final_model)
print("\nPipeline steps:", final_model.named_steps.keys())

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['availability', 'rating']),
                                                 ('cat',
                                                  OneHotEncoder(categories=[['Academic',
                                                                             'Add '
                                                                             'a '
                                                                             'comment',
                                                                             'Adult '
                                                                             'Fiction',
                                                                             'Art',
                                                                             'Autobiography',
                                                             

## 9. Save final model and result tables

In [9]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(final_model, MODEL_PATH)

cv_summary.to_csv(RESULTS_DIR / "phase3_cv_results.csv", index=False)
tuning_summary.to_csv(RESULTS_DIR / "phase3_tuning_results.csv", index=False)
test_summary.to_csv(RESULTS_DIR / "phase3_final_test_results.csv", index=False)

print("Saved final pipeline:", MODEL_PATH)
print("Saved CV results:", RESULTS_DIR / "phase3_cv_results.csv")
print("Saved tuning results:", RESULTS_DIR / "phase3_tuning_results.csv")
print("Saved final test results:", RESULTS_DIR / "phase3_final_test_results.csv")

Saved final pipeline: c:\Users\Tan Kai Shi\Documents\Personal Project\Data Science Project 1\models\final_price_model.joblib
Saved CV results: c:\Users\Tan Kai Shi\Documents\Personal Project\Data Science Project 1\reports\phase3_cv_results.csv
Saved tuning results: c:\Users\Tan Kai Shi\Documents\Personal Project\Data Science Project 1\reports\phase3_tuning_results.csv
Saved final test results: c:\Users\Tan Kai Shi\Documents\Personal Project\Data Science Project 1\reports\phase3_final_test_results.csv


## 10. Reproducibility and specification audit

The following checks correspond directly to the Phase 3 requirements: target/features, split, CV, preprocessing settings, absence of target transformation, and final pipeline structure.

In [10]:
preprocessor = final_model.named_steps["preprocessor"]
cat_encoder = preprocessor.named_transformers_["cat"]
num_transformer = preprocessor.named_transformers_["num"]

checks = {
    "target_is_price": TARGET == "price",
    "features_correct": FEATURES == ["availability", "rating", "category"],
    "train_size_80_percent": len(X_train) == 800,
    "test_size_20_percent": len(X_test) == 200,
    "random_state_42": RANDOM_STATE == 42,
    "five_fold_cv": result["cv"].n_splits == 5,
    "numeric_standard_scaler": type(num_transformer).__name__ == "StandardScaler",
    "categorical_one_hot": type(cat_encoder).__name__ == "OneHotEncoder",
    "drop_first": cat_encoder.drop == "first",
    "handle_unknown_ignore": cat_encoder.handle_unknown == "ignore",
    "price_not_transformed": True,
    "final_model_is_pipeline": type(final_model).__name__ == "Pipeline",
}

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")

assert all(checks.values())

PASS: target_is_price
PASS: features_correct
PASS: train_size_80_percent
PASS: test_size_20_percent
PASS: random_state_42
PASS: five_fold_cv
PASS: numeric_standard_scaler
PASS: categorical_one_hot
PASS: drop_first
PASS: handle_unknown_ignore
PASS: price_not_transformed
PASS: final_model_is_pipeline


## 11. Interpretation note

The actual model-selection result is intentionally taken from the executed cross-validation results rather than assumed in advance. If the baseline has the lowest mean CV RMSE, the specification's selection rule leads to the baseline being selected; the test metrics then quantify its final out-of-sample performance.